# Diffusion Policy on Push-T (ManiSkill)

**Runner notebook** — all logic lives in `/content/drive/MyDrive/continual_rl/`.

| Cell | What it does | Run again? |
|------|-------------|------------|
| 0 – Mount & paths | Mount Drive, add project to `sys.path` | Every restart |
| 1 – Install deps | Vulkan + pip (cached on Drive) | Every restart (~10 s after first run) |
| 2 – Sanity check | Quick env smoke-test | Optional |
| 3 – Download demos | Pull Push-T expert demos to Drive | Once ever |
| 4 – Convert demos | Convert to rgb + pd_ee_delta_pos format | Once ever |
| 5 – Configure | Build Config object; edit then re-run | Each experiment |
| 6 – Train | Full training loop; checkpoints → Drive | Per run |
| 7 – TensorBoard | Live loss/reward curves | Optional |
| 8 – Evaluate | Roll out best checkpoint | After training |
| 9 – Watch video | Display evaluation video | After eval |
| 10 – PPO baseline | State-based PPO for comparison | Optional |

## Cell 0 — Mount Drive & register project path
Run this **every** session restart. Nothing else happens until Drive is mounted.

In [8]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os

# ── edit this path if you placed the project elsewhere on Drive ──
PROJECT_DIR = '/content/drive/MyDrive/Continual-RL'
# ────────────────────────────────────────────────────────────────

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print('Drive mounted. Project root:', PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Project root: /content/drive/MyDrive/Continual-RL


## Cell 1 — Install dependencies
Packages are cached to Drive so **subsequent restarts take ~10 seconds** instead of 3+ minutes.

In [9]:
import importlib.util
import os
import sys

# Construct the full path to the intended install.py within the setup directory.
# PROJECT_DIR is defined in a previous cell and available in the kernel.
install_file_path = os.path.join(PROJECT_DIR, 'setup', 'install.py')

# Create a module specification from the file path, giving it a unique name
spec = importlib.util.spec_from_file_location("my_project_setup_install", install_file_path)

if spec is None:
    raise FileNotFoundError(
        f"Could not find the expected install.py at {install_file_path}. "
        "Please check if the file exists and PROJECT_DIR is correct."
    )

# Create a new module object from the specification
my_project_setup_install = importlib.util.module_from_spec(spec)

# Add the newly created module to sys.modules
sys.modules["my_project_setup_install"] = my_project_setup_install

# Execute the module to load its contents (e.g., functions, classes)
spec.loader.exec_module(my_project_setup_install)

# Now, full_setup can be accessed from the loaded module
full_setup = my_project_setup_install.full_setup

full_setup()  # Vulkan + pip install (Drive-cached)

[setup] Configuring Vulkan … done.
[setup] Installing packages to /content/drive/MyDrive/continual_rl_packages … 
[setup] Packages ready.
[setup] Environment ready. ✓


## Cell 2 — Sanity check: run one episode of Push-T

In [10]:
from envs import make_pusht_env
from utils import show_eval_grid
import torch

env = make_pusht_env(num_envs=4, obs_mode='rgb')
obs, _ = env.reset(seed=0)
env.unwrapped.print_sim_details()

for _ in range(10):
    action = torch.from_numpy(env.action_space.sample())
    obs, rew, term, trunc, info = env.step(action)

show_eval_grid(env, n_rows=2, n_cols=2, title='Push-T (4 parallel envs)')
env.close()
print('Sanity check passed ✓')

Download complete.
# -------------------------------------------------------------------------- #
Task ID: PushT-v1, 4 parallel environments, sim_backend=physx_cuda
obs_mode=rgb, control_mode=pd_ee_delta_pos
render_mode=rgb_array, sensor_details=RGBD(128x128)
sim_freq=100, control_freq=20
observation space: Dict('agent': Dict('qpos': Box(-inf, inf, (4, 7), float32), 'qvel': Box(-inf, inf, (4, 7), float32)), 'extra': Dict('tcp_pose': Box(-inf, inf, (4, 7), float32)), 'sensor_data': Dict('base_camera': Dict('rgb': Box(0, 255, (4, 128, 128, 3), uint8))), 'sensor_param': Dict('base_camera': Dict('cam2world_gl': Box(-inf, inf, (4, 4, 4), float32), 'extrinsic_cv': Box(-inf, inf, (4, 3, 4), float32), 'intrinsic_cv': Box(-inf, inf, (4, 3, 3), float32))))
(single) action space: Box(-1.0, 1.0, (3,), float32)
# -------------------------------------------------------------------------- #


AttributeError: 'TimeLimitWrapper' object has no attribute 'render_rgb_array'

## Cell 3 — Download Push-T demos to Drive
Skipped automatically if the file already exists on Drive.

In [ ]:
from data import download_demos

demo_dir = download_demos(
    env_id='PushT-v1',
    output_dir=f'{PROJECT_DIR}/demos',
)
print('Demo directory:', demo_dir)

## Cell 4 — Convert demos to RGB + pd_ee_delta_pos format
One-time step. Skipped on subsequent runs if the converted file exists.

In [ ]:
import os
from data.demo_loader import convert_demos

raw_traj = f'{PROJECT_DIR}/demos/PushT-v1/motionplanning/trajectory.h5'
converted_traj = raw_traj.replace(
    'trajectory.h5',
    'trajectory.rgb.pd_ee_delta_pos.cpu.h5'
)

if not os.path.exists(converted_traj):
    converted_traj = convert_demos(
        traj_path=raw_traj,
        obs_mode='rgb',
        control_mode='pd_ee_delta_pos',
        num_procs=2,
    )
else:
    print(f'Converted demos already exist: {converted_traj}')

print('Using:', converted_traj)

## Cell 5 — Configure the experiment
Edit values here, then re-run. Saved to Drive so you can reload any past config.

In [ ]:
from training.config import Config, EnvConfig, PolicyConfig, TrainConfig

cfg = Config(
    env=EnvConfig(
        env_id='PushT-v1',
        num_envs=64,
        eval_num_envs=10,
        obs_mode='rgb',
        image_size=96,
    ),
    policy=PolicyConfig(
        obs_horizon=2,
        pred_horizon=16,
        action_horizon=8,
        obs_cond_dim=256,
        num_diffusion_steps=100,
    ),
    train=TrainConfig(
        demo_path=converted_traj,
        normalizer_path=f'{PROJECT_DIR}/normalizer_stats.npz',
        num_epochs=100,
        batch_size=256,
        lr=1e-4,
        ckpt_dir=f'{PROJECT_DIR}/checkpoints',
        save_every=10,
        # resume_from=f'{PROJECT_DIR}/checkpoints/epoch_0050.pt',  # uncomment to resume
        eval_every=10,
        eval_video_dir=f'{PROJECT_DIR}/eval_videos',
        log_dir=f'{PROJECT_DIR}/runs',
    ),
)

cfg.save(f'{PROJECT_DIR}/config.json')
print(cfg)

## Cell 6 — Train Diffusion Policy
Checkpoints are saved to Drive every `save_every` epochs.
If the runtime dies, set `resume_from` in Cell 5 and re-run from there.

In [ ]:
from training.trainer import Trainer

trainer = Trainer(cfg)
trainer.train()

## Cell 7 — TensorBoard (live during training)
Open a separate tab and run this cell while training is running in Cell 6.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {cfg.train.log_dir}

Or plot the curve inline after training:

In [ ]:
from utils.visualization import plot_training_curve
plot_training_curve(cfg.train.log_dir)

## Cell 8 — Evaluate a saved checkpoint

In [ ]:
import os, glob
from training.trainer import Trainer
from training.config import Config

# Load config from Drive (in case kernel was restarted after training)
cfg = Config.load(f'{PROJECT_DIR}/config.json')

# Pick the latest checkpoint automatically
ckpts = sorted(glob.glob(f'{cfg.train.ckpt_dir}/epoch_*.pt'))
if not ckpts:
    raise FileNotFoundError('No checkpoints found. Run Cell 6 first.')

cfg.train.resume_from = ckpts[-1]
print('Evaluating checkpoint:', cfg.train.resume_from)

trainer = Trainer(cfg)
metrics = trainer.evaluate(epoch=int(ckpts[-1].split('epoch_')[1].split('.')[0]))
print('Metrics:', metrics)

## Cell 9 — Watch an evaluation video

In [ ]:
import glob
from utils.visualization import display_video

# Find the most recent eval video
videos = sorted(glob.glob(f'{cfg.train.eval_video_dir}/**/*.mp4', recursive=True))
if not videos:
    print('No eval videos yet. Run Cell 8 first.')
else:
    print('Showing:', videos[-1])
    display_video(videos[-1], width=640)

## Cell 10 — (Optional) State-based PPO baseline
Downloads ManiSkill's PPO script and runs it. Useful to compare against Diffusion Policy.
PPO files are cached to Drive so they're not re-downloaded on restart.

In [ ]:
import os

PPO_DIR = f'{PROJECT_DIR}/baselines'
os.makedirs(PPO_DIR, exist_ok=True)

for script in ['ppo.py', 'ppo_rgb.py']:
    dst = f'{PPO_DIR}/{script}'
    if not os.path.exists(dst):
        url = f'https://raw.githubusercontent.com/haosulab/ManiSkill/main/examples/baselines/ppo/{script}'
        os.system(f'wget -q {url} -O {dst}')
        print(f'Downloaded {script}')
    else:
        print(f'{script} already cached on Drive.')

In [ ]:
# State-based PPO — edit flags as needed
!python {PPO_DIR}/ppo.py \
    --env_id='PushT-v1' \
    --exp-name='state-pusht' \
    --num_envs=1024 \
    --update_epochs=8 \
    --num_minibatches=32 \
    --total_timesteps=600_000 \
    --eval_freq=8 \
    --num-steps=20

In [ ]:
import glob
from utils.visualization import display_video

ppo_videos = sorted(glob.glob('runs/state-pusht/videos/*.mp4'))
if ppo_videos:
    display_video(ppo_videos[-1], width=1024)

## Extra — Replay an expert demo

In [ ]:
from data import load_demo_metadata, replay_episode
from utils.visualization import display_video

h5, meta = load_demo_metadata(f'{PROJECT_DIR}/demos/PushT-v1/motionplanning/trajectory.h5')
print(f'{len(meta["episodes"])} demos available.')

video_path = replay_episode(
    episode_idx=0,
    h5_file=h5,
    json_data=meta,
    save_dir=f'{PROJECT_DIR}/replays',
)
display_video(video_path)